# PHASE 4: MODELING
## Project 7: Neural Network for Iris Classification

**Objective:** Design, train, and optimize a dense neural network for iris flower classification

**Status:** ✅ Complete - Phase 4 Modeling finished with 96.67% accuracy

---

# PROJECT 7: NEURAL NETWORK FOR IRIS CLASSIFICATION
## Module 8: Deep Learning Fundamentals

### SCOPE OF THIS PROJECT:

This project focuses **EXCLUSIVELY on Phase 4 (MODELING)** to understand the fundamentals of Deep Learning and Neural Networks (Dense Neural Networks).

#### What this project covers:
- ✅ Building Dense Neural Networks (DNN) from scratch
- ✅ Understanding layers, neurons, activation functions (ReLU, Softmax)
- ✅ Training with backpropagation and gradient descent
- ✅ Hyperparameter optimization: GridSearchCV + RandomizedSearchCV
- ✅ Evaluation metrics: Accuracy, F1-Score, Confusion Matrix
- ✅ Model convergence and overfitting detection

#### What this project does NOT cover:
- ❌ Full CRISP-DM cycle (Phases 1-3, 5-6 skipped)
- ❌ Exploratory Data Analysis (EDA)
- ❌ Feature engineering
- ❌ Convolutional Neural Networks (CNN)
- ❌ Recurrent Neural Networks (RNN)
- ❌ Production deployment

#### Dataset:
- **Source:** Iris (UCI Machine Learning Repository / Kaggle)
- **Size:** 150 samples, 4 features, 3 classes
- **Purpose:** Educational prototype for understanding DNN mechanics

#### Architecture Achieved:
Input(4) → Dense(16, ReLU) → Dense(4, ReLU) → Dense(3, Softmax)
- **Accuracy:** 96.67% on validation set
- **Error rate:** 1 out of 30 samples misclassified

**Author:** José Marcel López Pino | **Date:** March 2026 | **Bootcamp:** Fundamentos de Ciencia de Datos

In [ ]:
# ==============================================================================
# PHASE 4: MODELING - Neural Network for Iris Classification
# ==============================================================================

# ===== Standard Library =====
from pathlib import Path

# ===== Third-party: Data Science Stack (Core) =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ===== Third-party: Machine Learning =====
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    f1_score
)

# ===== Third-party: Deep Learning =====
from tensorflow import keras
from tensorflow.keras import layers

# ===== Third-party: Statistics =====
from scipy.stats import randint, uniform

# ===== Configuration: Reproducibility =====
SEED = 42
np.random.seed(SEED)
import tensorflow as tf
tf.random.set_seed(SEED)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 1. Load and Explore Iris Dataset

In [2]:
# Load Iris dataset from local CSV file
df = pd.read_csv("../data/raw/Iris.csv")

# Extract features (X)
feature_columns = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
X = df[feature_columns].values

# Encode target (y)
le = LabelEncoder()
y = le.fit_transform(df["Species"])
class_names = le.classes_

# Basic exploration
print("="*60)
print("IRIS DATASET - INITIAL EXPLORATION")
print("="*60)
print(f"Shape of X (features): {X.shape}")
print(f"Available classes: {class_names.tolist()}")
print(f"Numeric mapping: 0={class_names[0]}, 1={class_names[1]}, 2={class_names[2]}")
print(f"First 5 labels: {y[:5]}")
print(f"Class distribution: {np.bincount(y)}")
print()

IRIS DATASET - INITIAL EXPLORATION
Shape of X (features): (150, 4)
Available classes: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
Numeric mapping: 0=Iris-setosa, 1=Iris-versicolor, 2=Iris-virginica
First 5 labels: [0 0 0 0 0]
Class distribution: [50 50 50]



In [3]:
# ===== COMPREHENSIVE IRIS.CSV EXPLORATION =====

print("="*70)
print("IRIS.CSV - COMPREHENSIVE DATA EXPLORATION")
print("="*70)

# 1. Basic info
print("\n1. DATASET SHAPE")
print(f"   Rows (samples): {df.shape[0]}")
print(f"   Columns: {df.shape[1]}")

# 2. Column names and types
print("\n2. COLUMN NAMES & DATA TYPES")
print(df.dtypes)

# 3. First and last rows
print("\n3. FIRST 5 ROWS")
print(df.head())

print("\n4. LAST 5 ROWS")
print(df.tail())

# 5. Column statistics
print("\n5. DESCRIPTIVE STATISTICS (features only)")
print(df[feature_columns].describe())

# 6. Missing values
print("\n6. MISSING VALUES")
print(df.isnull().sum())

# 7. Unique species
print("\n7. SPECIES DISTRIBUTION")
print(df['Species'].value_counts())

# 8. Data types summary
print("\n8. DATA INFO")
print(df.info())

# 9. Memory usage
print("\n9. MEMORY USAGE")
print(df.memory_usage(deep=True))

print("\n" + "="*70)

IRIS.CSV - COMPREHENSIVE DATA EXPLORATION

1. DATASET SHAPE
   Rows (samples): 150
   Columns: 6

2. COLUMN NAMES & DATA TYPES
Id                 int64
SepalLengthCm    float64
SepalWidthCm     float64
PetalLengthCm    float64
PetalWidthCm     float64
Species              str
dtype: object

3. FIRST 5 ROWS
   Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm      Species
0   1            5.1           3.5            1.4           0.2  Iris-setosa
1   2            4.9           3.0            1.4           0.2  Iris-setosa
2   3            4.7           3.2            1.3           0.2  Iris-setosa
3   4            4.6           3.1            1.5           0.2  Iris-setosa
4   5            5.0           3.6            1.4           0.2  Iris-setosa

4. LAST 5 ROWS
      Id  SepalLengthCm  SepalWidthCm  PetalLengthCm  PetalWidthCm  \
145  146            6.7           3.0            5.2           2.3   
146  147            6.3           2.5            5.0           1.9   
147 

## 2. Train/Validation Split (80/20 stratified)

In [4]:
# Split data into train (80%) and validation (20%)
# stratify=y ensures each set maintains same class proportions
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("="*60)
print("TRAIN/VALIDATION SPLIT")
print("="*60)
print(f"Train shape: {X_train.shape} | {y_train.shape}")
print(f"Validation shape: {X_val.shape} | {y_val.shape}")
print(f"Train class distribution: {np.bincount(y_train)}")
print(f"Val class distribution: {np.bincount(y_val)}")
print()

TRAIN/VALIDATION SPLIT
Train shape: (120, 4) | (120,)
Validation shape: (30, 4) | (30,)
Train class distribution: [40 40 40]
Val class distribution: [10 10 10]



## 3. Feature Standardization (StandardScaler)

In [5]:
# Standardize features: mean=0, std=1
# Improves neural network convergence
# IMPORTANT: fit ONLY on train, transform on val (prevents data leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("="*60)
print("FEATURE STANDARDIZATION (STANDARDSCALER)")
print("="*60)
print(f"Train mean: {X_train_scaled.mean(axis=0).round(3)}")
print(f"Train std: {X_train_scaled.std(axis=0).round(3)}")
print("✅ Features standardized (mean≈0, std≈1)")
print()

FEATURE STANDARDIZATION (STANDARDSCALER)
Train mean: [-0. -0.  0.  0.]
Train std: [1. 1. 1. 1.]
✅ Features standardized (mean≈0, std≈1)



## 4. Function to Build Neural Network Model

In [6]:
def build_model(units_1=16, units_2=8, learning_rate=0.001):
    """
    Build sequential neural network for multiclass classification.
    
    Args:
        units_1 (int): Neurons in first hidden layer (default=16)
        units_2 (int): Neurons in second hidden layer (default=8)
        learning_rate (float): Learning rate for Adam optimizer (default=0.001)
    
    Returns:
        model (keras.Sequential): Compiled model ready for training
    """
    # Architecture:
    # Input(4) -> Dense(units_1, ReLU) -> Dense(units_2, ReLU) -> Dense(3, Softmax)
    model = keras.Sequential([
        layers.Input(shape=(4,)),  # 4 Iris features
        layers.Dense(units_1, activation="relu"),  # Hidden layer 1
        layers.Dense(units_2, activation="relu"),  # Hidden layer 2
        layers.Dense(3, activation="softmax")  # Output: 3 classes
    ])
    
    # Compile model
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    
    return model


# Create and display model architecture
model_baseline = build_model()
print("="*60)
print("NEURAL NETWORK ARCHITECTURE")
print("="*60)
model_baseline.summary()
print()

NEURAL NETWORK ARCHITECTURE


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)

## 5. Train Baseline Model (16, 8 neurons)

In [7]:
# Train model with default hyperparameters
print("="*60)
print("BASELINE TRAINING (20 epochs)")
print("="*60)
history_baseline = model_baseline.fit(
    X_train_scaled,
    y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_val_scaled, y_val),
    verbose=1
)

# Evaluate on validation
loss_baseline, acc_baseline = model_baseline.evaluate(
    X_val_scaled, y_val, verbose=0
)
print(f"\n✅ Baseline Accuracy: {acc_baseline:.4f}")
print()

BASELINE TRAINING (20 epochs)
Epoch 1/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.3000 - loss: 1.2912 - val_accuracy: 0.2667 - val_loss: 1.1989
Epoch 2/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2833 - loss: 1.2020 - val_accuracy: 0.3667 - val_loss: 1.1293
Epoch 3/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4333 - loss: 1.1223 - val_accuracy: 0.6000 - val_loss: 1.0653
Epoch 4/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5750 - loss: 1.0506 - val_accuracy: 0.6333 - val_loss: 1.0070
Epoch 5/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5500 - loss: 0.9863 - val_accuracy: 0.5000 - val_loss: 0.9532
Epoch 6/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5583 - loss: 0.9310 - val_accuracy: 0.4667 - val_loss: 0.9054
Epoch 7/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6000 - loss: 0.8817 - val_accuracy: 0.6000 - val_loss: 0.8610
Epoch 8/20
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6667 - loss: 0.8366 - val_accura

## 6. GridSearchCV: Exhaustive Hyperparameter Search

In [9]:
from scikeras.wrappers import KerasClassifier

# Wrapper to integrate Keras with sklearn
keras_clf = KerasClassifier(
    model=build_model,
    epochs=40,
    batch_size=16,
    verbose=0
)

# Define grid of hyperparameters to explore
param_grid = {
    "model__units_1": [8, 16],      # Layer 1 neurons
    "model__units_2": [4, 8],       # Layer 2 neurons
    "batch_size": [8, 16],          # Batch size
    "epochs": [40, 80]              # Training epochs
}

# Execute GridSearchCV (exhaustive search)
print("="*60)
print("GRIDSEARCHCV: Exhaustive Hyperparameter Search")
print("="*60)
print(f"Total combinations to test: {2*2*2*2} = 16 combinations")
print(f"Cross-validation: 3-fold")
print(f"Total trainings: 16 × 3 = 48 models")
print("\nProcessing... (this may take ~5-10 minutes)\n")

grid_search = GridSearchCV(
    estimator=keras_clf,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("\n" + "="*60)
print("GRIDSEARCHCV RESULTS")
print("="*60)
print(f"Best hyperparameters: {grid_search.best_params_}")
print(f"Best accuracy (CV): {grid_search.best_score_:.4f}")
print()

GRIDSEARCHCV: Exhaustive Hyperparameter Search
Total combinations to test: 16 = 16 combinations
Cross-validation: 3-fold
Total trainings: 16 × 3 = 48 models

Processing... (this may take ~5-10 minutes)

Fitting 3 folds for each of 16 candidates, totalling 48 fits

GRIDSEARCHCV RESULTS
Best hyperparameters: {'batch_size': 8, 'epochs': 80, 'model__units_1': 16, 'model__units_2': 4}
Best accuracy (CV): 0.9667



## 7. RandomizedSearchCV: Random Sampling of Hyperparameters

In [10]:
# Define distributions of hyperparameters
param_distributions = {
    "model__units_1": randint(8, 33),           # 8-32 neurons
    "model__units_2": randint(4, 17),           # 4-16 neurons
    "batch_size": [8, 16, 32],                  # Batch sizes
    "epochs": [40, 80, 120],                    # Epochs
    "model__learning_rate": uniform(1e-4, 9e-3)  # Learning rate
}

# Execute RandomizedSearchCV (random sampling)
print("="*60)
print("RANDOMIZEDSEARCHCV: Random Sampling of Hyperparameters")
print("="*60)
print(f"Iterations: 10 random samples")
print(f"Cross-validation: 3-fold")
print(f"Total trainings: 10 × 3 = 30 models")
print("\nProcessing... (this may take ~3-8 minutes)\n")

rand_search = RandomizedSearchCV(
    estimator=keras_clf,
    param_distributions=param_distributions,
    n_iter=10,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    random_state=SEED,
    verbose=1
)

rand_search.fit(X_train_scaled, y_train)

print("\n" + "="*60)
print("RANDOMIZEDSEARCHCV RESULTS")
print("="*60)
print(f"Best hyperparameters: {rand_search.best_params_}")
print(f"Best accuracy (CV): {rand_search.best_score_:.4f}")
print()

RANDOMIZEDSEARCHCV: Random Sampling of Hyperparameters
Iterations: 10 random samples
Cross-validation: 3-fold
Total trainings: 10 × 3 = 30 models

Processing... (this may take ~3-8 minutes)

Fitting 3 folds for each of 10 candidates, totalling 30 fits

RANDOMIZEDSEARCHCV RESULTS
Best hyperparameters: {'batch_size': 8, 'epochs': 120, 'model__learning_rate': np.float64(0.00411249477568232), 'model__units_1': 30, 'model__units_2': 14}
Best accuracy (CV): 0.9583



## 8. Train Final Model with Best Hyperparameters

In [11]:
# Use best parameters from GridSearchCV
best_params = grid_search.best_params_

# Build final model
final_model = build_model(
    units_1=best_params["model__units_1"],
    units_2=best_params["model__units_2"],
    learning_rate=best_params.get("model__learning_rate", 0.001)
)

# Train final model
print("="*60)
print("FINAL MODEL: Training with Best Hyperparameters")
print("="*60)
print(f"Configuration: units_1={best_params['model__units_1']}, "
      f"units_2={best_params['model__units_2']}, "
      f"batch_size={best_params['batch_size']}, "
      f"epochs={best_params['epochs']}")
print()

history_final = final_model.fit(
    X_train_scaled,
    y_train,
    epochs=best_params["epochs"],
    batch_size=best_params["batch_size"],
    validation_data=(X_val_scaled, y_val),
    verbose=1
)

# Evaluate final model
loss_final, acc_final = final_model.evaluate(
    X_val_scaled, y_val, verbose=0
)

print("\n" + "="*60)
print(f"FINAL MODEL ACCURACY: {acc_final:.4f}")
print("="*60)
print()

FINAL MODEL: Training with Best Hyperparameters
Configuration: units_1=16, units_2=4, batch_size=8, epochs=80

Epoch 1/80
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.3083 - loss: 1.2160 - val_accuracy: 0.2333 - val_loss: 1.1664
Epoch 2/80
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3083 - loss: 1.0935 - val_accuracy: 0.2333 - val_loss: 1.0648
Epoch 3/80
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3750 - loss: 1.0098 - val_accuracy: 0.3333 - val_loss: 1.0050
Epoch 4/80
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4000 - loss: 0.9640 - val_accuracy: 0.4000 - val_loss: 0.9646
Epoch 5/80
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4250 - loss: 0.9289 - val_accuracy: 0.4333 - val_loss: 0.9299
Epoch 6/80
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4250 - loss: 0.8981 - val_accuracy: 0.4333 - val_loss: 0.8983
Epoch 7/80
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4083 - loss: 0.8698 - val_accuracy: 0.4333 - val_loss: 0.8708
Epoch 

## 9. Detailed Evaluation: Metrics and Visualizations

In [13]:
# Predictions on validation
y_pred_proba = final_model.predict(X_val_scaled)
y_pred = y_pred_proba.argmax(axis=1)

# Confusion matrix
cm = confusion_matrix(y_val, y_pred)
f1_avg = f1_score(y_val, y_pred, average="weighted")

print("="*60)
print("EVALUATION METRICS")
print("="*60)
print(f"Accuracy: {acc_final:.4f}")
print(f"F1-Score (weighted): {f1_avg:.4f}")
print()
print("Confusion Matrix:")
print(cm)
print()
print("Classification Report:")
print(classification_report(
    y_val,
    y_pred,
    target_names=class_names
))
print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
EVALUATION METRICS
Accuracy: 0.9667
F1-Score (weighted): 0.9666

Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

Classification Report:
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.90      0.95        10
 Iris-virginica       0.91      1.00      0.95        10

       accuracy                           0.97        30
      macro avg       0.97      0.97      0.97        30
   weighted avg       0.97      0.97      0.97        30




## 10. Summary and Conclusions

In [14]:
print("="*60)
print("SUMMARY: PHASE 4 - MODELING")
print("="*60)
print(f"Baseline Accuracy: {acc_baseline:.4f}")
print(f"GridSearchCV Accuracy: {grid_search.best_score_:.4f}")
print(f"RandomizedSearchCV Accuracy: {rand_search.best_score_:.4f}")
print(f"Final Model Accuracy: {acc_final:.4f}")
print()
print("Best configuration found (GridSearchCV):")
print(f"  - Hidden units 1: {best_params['model__units_1']}")
print(f"  - Hidden units 2: {best_params['model__units_2']}")
print(f"  - Batch size: {best_params['batch_size']}")
print(f"  - Epochs: {best_params['epochs']}")
print()
print("✅ PHASE 4 COMPLETE - Proceed to PHASE 5: EVALUATION")

SUMMARY: PHASE 4 - MODELING
Baseline Accuracy: 0.8000
GridSearchCV Accuracy: 0.9667
RandomizedSearchCV Accuracy: 0.9583
Final Model Accuracy: 0.9667

Best configuration found (GridSearchCV):
  - Hidden units 1: 16
  - Hidden units 2: 4
  - Batch size: 8
  - Epochs: 80

✅ PHASE 4 COMPLETE - Proceed to PHASE 5: EVALUATION


## Next Steps — Project 7B & 8

**Project Scope Clarification:**

This Project 7 notebook is a **simulation and learning exercise** focused exclusively on understanding how Dense Neural Networks (DNN) work mechanically:
- How layers, neurons, and activation functions transform data
- How backpropagation and gradient descent optimize weights
- How hyperparameter tuning (GridSearchCV, RandomizedSearchCV) improves performance
- How to measure model quality (accuracy, F1-score, confusion matrix)

**This is NOT a full CRISP-DM project** — it skips Phases 1-3 and 5-6 because the Iris dataset is already clean, well-understood, and used purely for educational purposes.

**Immediate Next (Project 7B):**
- Build Convolutional Neural Networks (CNN) for image classification
- Dataset: Fashion-MNIST (70k clothing images, 10 categories: t-shirt, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot)
- Goal: Understand CNN architecture (Conv2D, MaxPooling, Dense) for visual feature extraction
- Apply Deep Learning fundamentals from Project 7 to image data

**Future Application (Post-Bootcamp):**
- Transfer Learning on real PequeShop children's clothing dataset (5-10 years)
- Custom CNN trained on actual product images
- Production deployment for e-commerce platform

**Medium Term (Project 8):**
- Apache Spark + Big Data processing
- Scalable ML pipelines with MLlib
- Real-world retail analytics at scale